In [19]:
# Импортируем необходимые библиотеки
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import re

## Задание 1.
Реализуйте базовый класс Account, который моделирует поведение
банковского счёта. Этот класс должен не только выполнять базовые
операции, но и вести детальный учёт всех действий, а также предоставлять
аналитику по истории операций.

### Этап 1.
Реализация базового класса Account
Класс должен быть инициализирован с параметрами:
- account_holder (str) — имя владельца счёта;
- balance (float, по умолчанию 0) — начальный баланс счёта, не может
быть отрицательным.
Атрибуты:
- _account_counter — приватный атрибут для хранения количества
созданных счетов. Отсчет начинается с 1000;
- holder — хранит имя владельца;
- account_number — хранит номер счёта;
- _balance — приватный атрибут для хранения текущего баланса;
- operations_history — список или другая структура для хранения
истории операций.
Важно: каждая операция должна храниться не просто как число, а как
структурированная информация, например, словарь, кортеж или класс.
Минимальный набор данных для операции: тип операции ('deposit' или
'withdraw'), сумма, дата и время операции, текущий баланс после операции,
статус ('success' или 'fail').

### Этап 2.
Реализация методов
1. __init__(self, account_holder, balance=0) — конструктор. Обратите
внимание, что в конструкторе должен автоматически формироваться
номер счёта в формате ‘ACC-XXXX’, где XXXX — порядковый номер
счёта;
2. deposit(self, amount) — метод для пополнения счёта:
    - принимает сумму (должна быть положительной), попытка
    положить отрицательную сумму, должна вызывать исключение;
    - в случае успеха обновляет баланс и добавляет запись в историю
    операций.
3. withdraw(self, amount) — метод для снятия средств:
    - принимает сумму (должна быть положительной);
    - проверяет, достаточно ли средств на счёте, если нет — операция
    не проходит, но ее попытка с статусом 'fail' все равно фиксируется
    в истории;
    - в случае успеха обновляет баланс и добавляет запись в историю.
4.  get_balance(self) — метод, который возвращает текущий баланс.
5.  get_history(self) — метод, который возвращает историю операций.
Важно: продумайте, в каком формате его вернуть. Для работы с датой и
временем используйте модуль datetime. Получить текущее время можно с
помощью datetime.now().

Важно: продумайте, в каком формате его вернуть. Для работы с датой и
временем используйте модуль datetime. Получить текущее время можно с
помощью datetime.now().

### Этап 3. Визуализация истории операций. Дополнительное задание для
претендующих на оценку 8 и выше баллов (выполняется по желанию).
1. Создайте метод plot_history(self), который использует библиотеку
Pandas для создания датафрейма из истории операций.
2. Продумайте, с помощью какой библиотеки можно отобразить
изменение баланса с течением времени. Постройте простой
линейный график, где по оси X будет время операции, а по оси Y —
баланс после каждой операции. График должен иметь заголовок,
подписи осей.

## Задание 2.
### Этап 4. Реализация наследования
1. Реализуйте два класса CheckingAccount (расчётный счёт) и
SavingsAccount (сберегательный счёт), которые отражают абстракцию
базового поведения банковских аккаунтов:
    - наследуются от базового класса Account;
    - хранят атрибут класса account_type.
2. Класс SavingsAccount (сберегательный счёт) дополнительно должен
реализовывать метод расчёта процентов на остаток
apply_interest(self, rate) (например, 7% на остаток).
3. Класс SavingsAccount (сберегательный счёт) позволяет снимать
деньги только до определенного порога баланса: нельзя снять
больше 50% от баланса. Переопределите метод снятия со счёта.
4. Реализуйте валидацию на отрицательные суммы и корректность
имени владельца:
    - имя владельца счёта должно быть в формате «Имя Фамилия» с
заглавных букв, кириллицей или латиницей, иначе — должно
вызываться исключение;
    - попытка положить отрицательную сумму должна вызывать
исключение.
5. Реализуйте метод для анализа истории транзакций по размеру и дате:
    - метод должен выводить последние n крупных операций.

In [20]:
class Account:
    # Приватный атрибут для хранения созданных счетов. Отсчет начинается с 1000.
    _account_counter = 1000

    def __init__(self, account_holder: str, balance: float = 0.0):
        # Проверка имени владельцы
        if not self._validate_holder(account_holder):
            raise ValueError("Имя владельца должно быть в формате 'Имя Фамилия' (с заглавных букв, кириллицей или латиницей).")
        # Проверка на отрицательные суммы
        if balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным.")

        self.holder = account_holder # Хранит имя владельца
        self._balance = balance      # Приватный атрибут для хранения текущего баланса

        Account._account_counter += 1                           # Каждый раз при инициализации увеличиваем номер счета на 1
        self.account_number = f'ACC-{Account._account_counter}' # Хранит номер счета формата 'ACC-XXXX'

        self.operations_history = [] # Список для хранения истории операций

    # Проверка корректности имени владельца
    @staticmethod
    def _validate_holder(holder: str):
        pattern = r"^[A-ZА-ЯЁ][a-zа-яё]+ [A-ZА-ЯЁ][a-zа-яё]+$"
        return bool(re.match(pattern, holder))

    def deposit(self, amount: float):
        """Метод пополнения счета"""
        if amount <= 0:
            # Сумма депозита должна быть положительная. Вызываем исключение при отрицательной сумме.
            message = f"Сумма должна быть положительной. Текущая сумма: {amount}"
            raise ValueError(message)
        else:
            # Обновляем баланс
            self._balance += amount
            status = "success"
            message = f"Пополнение на {amount:.2f}"

        # Записываем историю операции
        self._record_operation("deposit", amount, status, message)
        return status

    def withdraw(self, amount: float):
        """Метод снятия средств"""
        if amount <= 0:
            message = f"Сумма должна быть положительной. Текущая сумма: {amount}"
            raise ValueError(message)
        elif amount > self._balance:
            status = "fail"
            message = "Недостаточно средств"
        else:
            self._balance -= amount
            status = 'success'
            message = f"Снятие {amount:.2f}"

        # Записываем историю операции
        self._record_operation("withdraw", amount, status, message)
        return status

    # Технический метод сохранения истории операции
    def _record_operation(self, operation_type: str, amount: float, status: str, message: str = ""):
        self.operations_history.append({
            'operation_type': operation_type,
            'amount': amount,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'balance_after_operation': self._balance,
            'status': status,
            'message': message
        })

    def get_balance(self):
        """Метод, который возвращает текущий баланс."""
        return self._balance

    def get_history(self):
        """Метод, который возвращает историю операций."""
        return self.operations_history.copy()

    def plot_history(self):
        """Метод, который строит график изменения баланса"""
        # Проверяем, что история операций не пуста
        if not self.operations_history:
            print("История пуста, график построить нельзя.")
            return

        # Создаем DataFrame из истории операций
        df = pd.DataFrame(self.operations_history)
        # Приводим к datetime
        df['timestamp'] = pd.to_datetime(df['timestamp'])

        # Построение графика изменения баланса
        plt.figure(figsize=(8, 5))
        plt.plot(df["timestamp"], df["balance_after_operation"])

        plt.title(f"Изменение баланса по счету {self.account_number} ({self.holder})")
        plt.xlabel("Время операции")
        plt.ylabel("Баланс после операции")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    def analyze_operations(self, n=5):
        """Показать последние n крупных операций по размеру."""
        if not self.operations_history:
            print("История пуста.")
            return
        df = pd.DataFrame(self.operations_history)
        df_success = df[df['status'] == 'success']
        df_sorted = df_success.sort_values(by="amount", ascending=False).head(n)
        print(df_sorted[["timestamp", "operation_type", "amount", "status", "message"]])

    def __str__(self):
        return f"Account({self.account_number}) — {self.holder}, баланс: {self._balance:.2f}₽"


In [21]:
# Класс расчетного счета
class CheckingAccount(Account):
    account_type = 'Checking'

    def __init__(self, account_holder: str, balance: float = 0.0):
        super().__init__(account_holder, balance)

# Класс сберегательного счета
class SavingsAccount(Account):
    account_type = 'Savings'

    def __init__(self, account_holder: str, balance: float = 0.0):
        super().__init__(account_holder, balance)

    def apply_interest(self, rate: float):
        """Метод начислений процентов на остаток"""
        if rate < 0:
            raise ValueError("Процентная ставка должна быть положительной")

        interest = self._balance * (rate / 100)
        self._balance += interest
        self._record_operation("interest", interest, "success", f"Начислены проценты {rate:.2f}%")
        return interest

    def withdraw(self, amount: float):
        '''Нельзя снять больше 50% от баланса'''
        if amount > self._balance * 0.5:
            self._record_operation('withdraw', amount, 'fail', 'Превышен лимит снятия (50% от баланса)')
            return 'fail'
        return super().withdraw(amount)